<a href="https://colab.research.google.com/github/ibnu-ahmedin/afaan-oromo-hybrid-sentiment-analysis/blob/main/SVM_char.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# SVM CHAR N-GRAMS
# ==========================================================

import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Load splits
train_df = pd.read_csv('/content/train.csv')
val_df   = pd.read_csv('/content/val.csv')
test_df  = pd.read_csv('/content/test.csv')

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

afan_oromo_stopwords = {
                 "fi","kan","akka",'akka','waan','isa','kan','keessa','irra','itti',
                 'aanee','gidduu','narraa','gubbaa','ittuu','natti','akkam','hanga','hamma','jala',
                 'nu','nurraa','jara','hennaa','akkasumas','akkuma','hogguu','sana','nuti','ala',
                 'illee','siin','alatti','immoo','kana','silaa','sila','amma','inni','kanaafi',
                 'kanaaf','sitti','ammo','kanaafuu','sun','an','ani','isaaf','keenna','tanaafuu',
                 'ati','isaanirraa','dha','bira','isatti','keessan','teenya','booda','tun','ishiirraa',
                 'kun','yommuu','keenya','utuu','duuba','koo','yeroo','moo','eega','eegasii',
                 'isii','malee','booddee','dura','ishiif','yoo','fi','isin','na','yookaan',
                 'gama','isiin','isheen','ishii','naaf','isaanif','anaaf','erge','isheerraa','isinirraa',
                 'garuu','yookiin','keessatti','warra','yoom','anarraa','sirraa','ykn','fakkeenyaaf',"ta'a",
                 'kiyya','kanamalees','jidduu'"haa", "ture", "turte",  "jira", "jiru", "jirtu", "jedha",
                 "jedhe", "jedhan"
}

# ==========================================================
# 🔥 2. TEXT PREPROCESSING
# ==========================================================


def clean_text(text):
  if pd.isna(text):
        return ""
  text = str(text).lower()                               # Lowercase
  text = re.sub(r'http\S+|www\S+', '', text)             # Remove URLs
  text = re.sub(r'@\w+', '@user', text)                  # Normalize user mentions
  text = re.sub(r'[\U0001F600-\U0001FFFF]', '', text)    # Remove all emojis
  text = re.sub(r'(.)\1{2,}', r'\1\1', text)             # Collapse repeated letters
  text = re.sub(r"[^\w\s']", '', text)                   # Remove punctuation BUT KEEP apostrophes
  text = ' '.join(text.split())                          # Normalize spaces

  return text

# Apply cleaning
train_df["clean_text"] = train_df["Text"].apply(clean_text)
test_df["clean_text"]  = test_df["Text"].apply(clean_text)

safe_stopwords = afan_oromo_stopwords - {"hin", "miti"}

def remove_stopwords(text):
    return " ".join([w for w in text.split() if w not in safe_stopwords])

train_df["clean_text"] = train_df["clean_text"].apply(remove_stopwords)
test_df["clean_text"]  = test_df["clean_text"].apply(remove_stopwords)
# TF-IDF CHAR
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(3,5), max_features=10000)
X_train = vectorizer.fit_transform(train_df["clean_text"])
X_test  = vectorizer.transform(test_df["clean_text"])

y_train, y_test = train_df["label"], test_df["label"]

# Train
model = LinearSVC(max_iter=3000)
model.fit(X_train, y_train)

# Evaluate
preds = model.predict(X_test)

print("\nSVM CHAR RESULTS")
print("Accuracy:", round(accuracy_score(y_test, preds), 4))
print(classification_report(y_test, preds, target_names=["Negative","Neutral","Positive"]))

cm = confusion_matrix(y_test, preds)
ConfusionMatrixDisplay(cm, display_labels=["Negative","Neutral","Positive"]).plot(cmap="Blues")
plt.title("SVM Char")
plt.show()